# ⚡ Open Generative AI — 100% Free Cloud GPU (Google Colab)
### Free NVIDIA T4 GPU (16GB VRAM) • Zero Content Filters • Uncensored Fast Generation

👉 **Step 1:** Top menu me **Runtime → Change runtime type → T4 GPU** select karein.
👉 **Step 2:** Neeche diye gaye cells ko run karein (Play button ▶️ par click karein).

In [ ]:
#@title 1. GPU Check & Dependencies Install
!nvidia-smi
!pip install -q diffusers transformers accelerate torch torchvision torchaudio fastapi uvicorn pydantic pyngrok gradio nest_asyncio

In [ ]:
#@title 2. Launch 100% Uncensored Cloud GPU Studio (with Free Public Link)
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import gradio as gr

print("[*] Loading DreamShaper 8 (Uncensored SD 1.5) on GPU...")
model_id = "Lykon/DreamShaper"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
pipe.enable_attention_slicing()
print("[*] Model loaded successfully on GPU!")

def generate_image(prompt, negative_prompt, steps, guidance_scale, width, height, seed):
    generator = None
    if seed != -1:
        generator = torch.Generator("cuda").manual_seed(int(seed))
    else:
        seed = int(torch.randint(0, 2147483647, (1,)).item())
        generator = torch.Generator("cuda").manual_seed(seed)
        
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        width=int(width),
        height=int(height),
        generator=generator
    ).images[0]
    
    return image, f"Generated with Seed: {seed}"

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="Open Generative AI Cloud GPU") as demo:
    gr.Markdown("# ⚡ Open Generative AI — Cloud GPU Studio")
    gr.Markdown("**100% Free NVIDIA T4 GPU • No Filters • Full Creative Freedom**")
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", placeholder="Describe what you want to generate...", lines=3, value="a beautiful stunning portrait of a gorgeous woman, masterpiece, 8k uhd, cinematic lighting")
            negative_prompt = gr.Textbox(label="Negative Prompt", lines=2, value="ugly, blurry, deformed hands, bad anatomy, low quality, cartoon")
            
            with gr.Row():
                steps = gr.Slider(minimum=10, maximum=50, value=25, step=1, label="Sampling Steps")
                guidance_scale = gr.Slider(minimum=1.0, maximum=15.0, value=7.5, step=0.5, label="CFG Scale")
                
            with gr.Row():
                width = gr.Dropdown(choices=[512, 768], value=512, label="Width")
                height = gr.Dropdown(choices=[512, 768], value=512, label="Height")
                seed = gr.Number(value=-1, label="Seed (-1 for random)")
                
            generate_btn = gr.Button("🚀 Generate Image (Cloud GPU)", variant="primary")
            info_text = gr.Markdown("")
            
        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", type="pil")
            
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt, negative_prompt, steps, guidance_scale, width, height, seed],
        outputs=[output_image, info_text]
    )

demo.launch(share=True, debug=True)